# Few-shot IGT with Bolmo 1B (Ainu / Jejueo / Tsez)

Same two settings as the zero-shot notebook (`zero_shot_bolmo.ipynb`), both **one model call**:
- **all-in-one**: predict transliteration and gloss together
- **step-by-step**: the same prompt asks for Step 1 (transliteration), then Step 2 (gloss based on Step 1)

The only difference: the prompt first shows `SHOTS` solved examples from the **train** split (gold transliteration and gloss). The examples are fixed (the same for every test row, chosen with seed 42, see `few_shot` in `configs/zero_shot.json`), and no example text occurs in the dev or test split. Logic: `zero_shot/run_zero_shot.py --shots`.

Before running: **Runtime -> Change runtime type -> T4 GPU**.

## Resumable test run (recommended for the full test split)

Runs both settings as **background processes** on the Colab machine and saves every finished row to **Google Drive** (`MyDrive/korean-igt-few-shot/<lang>/`). A closed browser does not stop the run, and if the runtime is lost, running cells A and B again (in a new runtime) **continues where it stopped** (`--resume`). Greedy decoding makes the resumed rows identical to an uninterrupted run.

You don't need sections 1–6 for this. Choose the language in cell B. Cell B refuses to start a second copy while one is running.

### A. Connect Google Drive (asks for permission once per runtime)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### B. Start (or continue) the test runs

In [ ]:
LANG_RUN = "ainu"   # @param ["ainu", "jejueo", "tsez"]
SHOTS_RUN = 3       # @param {type:"integer"}

import os, subprocess
REPO = "/content/Korean-IGT-Bolmo"
OUT = f"/content/drive/MyDrive/korean-igt-few-shot/{LANG_RUN}"
os.makedirs(OUT, exist_ok=True)
setup = (f"cd /content && (test -d {REPO} || git clone -q https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git)"
         f" && cd {REPO} && git pull -q && git log --oneline -1"
         " && pip install -q 'transformers==4.57.3' 'xlstm==2.0.4' datasets pandas accelerate")
r = subprocess.run(setup, shell=True, capture_output=True, text=True)
print(r.stdout.strip(), r.stderr.strip()[-300:])

running = subprocess.run(["pgrep", "-f", "run_zero_shot.py"], capture_output=True, text=True).stdout.split()
if running:
    print("Already running (pids", running, "), not starting a second copy.")
else:
    for i, strategy in enumerate(("all_in_one", "step_by_step")):
        # the second run waits 3 min so the two runs do not download the model at the same time
        cmd = (f"sleep {180 * i}; cd {REPO} && python zero_shot/run_zero_shot.py --lang {LANG_RUN} "
               f"--split test --shots {SHOTS_RUN} --strategy {strategy} --resume --out-dir {OUT} "
               f">> {OUT}/{strategy}.log 2>&1")
        subprocess.Popen(["nohup", "bash", "-c", cmd], stdout=subprocess.DEVNULL,
                         stderr=subprocess.DEVNULL, start_new_session=True)
    print(f"Started {LANG_RUN} {SHOTS_RUN}-shot test runs (all_in_one, step_by_step) -> {OUT}")

### C. Progress (run any time; it only reads files)

In [ ]:
import json, os, subprocess
REPO = "/content/Korean-IGT-Bolmo"
OUT = f"/content/drive/MyDrive/korean-igt-few-shot/{LANG_RUN}"
cfg = json.load(open(f"{REPO}/configs/zero_shot.json"))
manifest = json.load(open(f"{REPO}/{cfg['languages'][LANG_RUN]['dataset_manifest']}"))
total = manifest["split_counts"]["test"]
for strategy in ("all_in_one", "step_by_step"):
    stem = f"{OUT}/bolmo-1b_{strategy}_{SHOTS_RUN}shot_test"
    n = 0
    if os.path.exists(stem + ".partial.jsonl"):
        with open(stem + ".partial.jsonl", encoding="utf-8") as fh:
            n = max(0, sum(1 for line in fh if line.strip()) - 1)
    state = "FINISHED" if os.path.exists(stem + ".csv") else "running or stopped"
    print(f"{LANG_RUN} {strategy}: {n}/{total} rows saved  [{state}]")
running = subprocess.run(["pgrep", "-f", "run_zero_shot.py"], capture_output=True, text=True).stdout.split()
print("processes running:", len(running), "(0 and not FINISHED = stopped: run cell B again to continue)")

### D. Download the finished results

In [ ]:
import glob, zipfile
from google.colab import files
zip_path = f"/content/few_shot_{LANG_RUN}_{SHOTS_RUN}shot_results.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(f"{OUT}/bolmo-1b_*_{SHOTS_RUN}shot_test.*"):
        z.write(f, arcname=os.path.basename(f))
print(sorted(zipfile.ZipFile(zip_path).namelist()))
files.download(zip_path)

## 1. Install (once per runtime), then Runtime -> Restart session

In [ ]:
!pip install "transformers==4.57.3" "xlstm==2.0.4" datasets pandas accelerate

## 2. Get the code and check the environment

In [ ]:
import os, sys, importlib, importlib.util
BRANCH = "main"
REPO = "/content/Korean-IGT-Bolmo"
if not os.path.isdir(REPO):
    !git clone https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git {REPO}
!git -C {REPO} fetch origin
!git -C {REPO} checkout {BRANCH}
!git -C {REPO} pull origin {BRANCH}
os.chdir(REPO)
sys.path.insert(0, REPO + "/zero_shot")
!git log --oneline -1

import transformers
print("transformers:", transformers.__version__, "(need 4.57.3)")
print("xlstm installed:", importlib.util.find_spec("xlstm") is not None)

## 3. Choose the language and the number of examples, and preview the prompts (no model needed)

In [ ]:
LANG = "jejueo"   # "ainu", "jejueo" or "tsez"
SHOTS = 3         # number of train examples in the prompt

import json, pandas as pd
import run_zero_shot as rz
importlib.reload(rz)
cfg = json.load(open("configs/zero_shot.json"))
F = cfg["few_shot"]
L = cfg["languages"][LANG]
manifest = rz.ROOT / L["dataset_manifest"]
name = L["language_name"]
eval_texts = [r["text"] for sp in ("dev", "test") for r in rz.load_rows(sp, None, manifest)]
examples = rz.select_examples(rz.load_rows("train", None, manifest), SHOTS, F["seed"], eval_texts,
                              F["max_text_chars"], F["max_gloss_chars"], F["min_text_words"])
block = rz.examples_block(examples, name)
T = {k: rz.add_examples(rz.fill_language(rz.load_prompt(cfg["prompts"][k]), name), block, name)
     for k in ("all_in_one", "step_by_step")}
ex = rz.load_rows("dev", 1, manifest)[0]
for key in ("all_in_one", "step_by_step"):
    print(f"----- {key} -----")
    print(rz.render(T[key], original_script=ex["text"], translation=ex["translation"]))
print("----- gold -----")
print(ex["transcription"], "|", ex["glosses"])

## 4. Smoke test: 10 dev rows, both settings

This only checks the pipeline. It is **not** a result. The examples are the same as in the test run.

In [ ]:
pd.set_option("display.max_colwidth", 80)
smoke = rz.main(["--lang", LANG, "--split", "dev", "--limit", "10", "--shots", str(SHOTS)])
for p in smoke:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print()
    print("=====", p.name, "=====")
    print(df.parse_status.value_counts().to_dict(), "| truncated:", int(df.truncated.sum()),
          "| seconds:", meta["runtime_seconds"], "| mean new tokens:", round(df.n_new_tokens.mean()))
    display(df[["gold_transliteration", "pred_transliteration", "gold_gloss", "pred_gloss", "parse_status"]])
    display(df[["raw_output"]])

## 5. Full run on the test split: all-in-one

Run this only after the smoke test looks right. Do not change prompts after looking at test outputs.

Time on a T4: about **2.5 hours (Ainu) to 5 hours (Jejueo)** (the 10-row Jejueo dev smoke test took about 2.5 min per setting, zero-shot 45 s, mostly because of repetition loops). Keep the tab open and the computer awake. The two settings run in separate cells so that a disconnect loses at most one of them.

In [ ]:
full = rz.main(["--lang", LANG, "--split", "test", "--shots", str(SHOTS), "--strategy", "all_in_one"])
for p in full:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print(p.name, "| rows:", len(df), "|", df.parse_status.value_counts().to_dict(),
          "| truncated:", int(df.truncated.sum()), "| minutes:", round(meta["runtime_seconds"] / 60, 1))

### Download the all-in-one results (right after 5 finishes)

In [ ]:
out_dir = F["output_dir"].format(lang=LANG)
zip_path = f"/content/few_shot_{LANG}_{SHOTS}shot_all_in_one_results.zip"
!zip -r {zip_path} {out_dir}
from google.colab import files
files.download(zip_path)

## 6. Full run on the test split: step-by-step

About **2.5 hours (Ainu) to 5 hours (Jejueo)** on a T4. It can be a new session: run sections 1–3 again first (install, restart, get the code, choose `LANG` and `SHOTS`), then this cell.

In [ ]:
full = rz.main(["--lang", LANG, "--split", "test", "--shots", str(SHOTS), "--strategy", "step_by_step"])
for p in full:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print(p.name, "| rows:", len(df), "|", df.parse_status.value_counts().to_dict(),
          "| truncated:", int(df.truncated.sum()), "| minutes:", round(meta["runtime_seconds"] / 60, 1))

### Download the step-by-step results (right after 6 finishes)

In [ ]:
out_dir = F["output_dir"].format(lang=LANG)
zip_path = f"/content/few_shot_{LANG}_{SHOTS}shot_step_by_step_results.zip"
!zip -r {zip_path} {out_dir}
from google.colab import files
files.download(zip_path)